# SLAM-ASR: Google Drive Checkpoint Değerlendirme Aracı (Mozilla Common Voice)
### Eğitilmiş Tüm Modellerin Common Voice (CV-TR) Üzerinde Hızlı Çapraz Korpus Testi

Bu notebook, Google Drive'ınızda (`/content/drive/MyDrive/checkpoints/`) kayıtlı olan tüm SLAM-ASR Türkçe modellerini (`t0`, `t1-mlp-lora`, `t1-lora`, `t1-tail-aug-lora`, `t1-tail-aug`, `t3-ng-ta-gormez` vb.) **yeniden eğitime gerek kalmadan** Mozilla Common Voice Türkçe test seti üzerinde hızlıca değerlendirmek için tasarlanmıştır.

* **Yeniden Eğitim:** Gerekmez (Ağırlıklar doğrudan Drive'dan yüklenir).
* **Değerlendirme Süresi:** Model başına yaklaşık ~6-7 dakika (GPU ile).
* **Eski Klasörü:** Drive'daki `Eski/` arşiv klasörü otomatik olarak göz ardı edilir.
* **Otomatik Kayıt:** Hesaplanan `eval_results_cv_generalization.jsonl` sonuçları doğrudan Google Drive'daki ilgili model klasörüne kalıcı olarak kaydedilir.


In [ ]:
# 1. GPU Kontrolü ve Google Drive Bağlantısı
!nvidia-smi

import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("[*] Google Drive başarıyla bağlandı.")
else:
    print("[*] Google Drive zaten bağlı.")


In [ ]:
# 2. Çalışma Dizinini Hazırlama ve Bağımlılıkları Yükleme
import os

if not os.path.exists("slam-asr-reimplement"):
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git

# Tüm projektör mimarilerini destekleyen t5 dizinine geçin
%cd /content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora

!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate datacollective mutagen peft


In [ ]:
# 3. Mozilla Common Voice Test Verisini Hazırlama (Tek Seferlik)
import os
import json
from google.colab import userdata

cv_test_manifest = "data/cv/test.jsonl"

def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return bool(audio_path and os.path.exists(audio_path))
    except Exception:
        return False
    return False

if not is_valid_manifest(cv_test_manifest):
    mdc_api_key = os.environ.get("MDC_API_KEY")
    if not mdc_api_key:
        try:
            mdc_api_key = userdata.get("MDC_API_KEY")
        except Exception:
            mdc_api_key = input("Lütfen Mozilla Data Collective API anahtarınızı (MDC_API_KEY) girin: ")
    
    print("[*] Common Voice test verisi indiriliyor ve hazırlanıyor...")
    !python scripts/prepare_common_voice.py \
        --download \
        --api_key "{mdc_api_key}" \
        --output_dir data/cv
else:
    print(f"[*] Common Voice test manifestosu ve ses dosyaları hazır: {cv_test_manifest}")


In [ ]:
# 4. Google Drive'daki Mevcut TSC Kontrol Noktalarını Tarama ('Eski' Klasörü Hariç)
import glob
import os
import re
import json

checkpoints_root = "/content/drive/MyDrive/checkpoints"

def extract_model_tag(folder_name):
    # e.g. 20261018_170924_slam_asr_tr_tsc_t0 -> t0
    # e.g. 20261021_045752_slam_asr_tr_tsc_t1_mlp_lora -> t1-mlp-lora
    # e.g. 20261018_170924_slam_asr_tr_tsc -> t0
    m = re.search(r"slam_asr_tr_tsc_?(.*)", folder_name)
    if m and m.group(1):
        tag = m.group(1).replace("_", "-").strip("-")
        return tag if tag else "t0"
    return "t0"

# 1. Tüm adayları bul ve 'Eski' / 'eski' klasörünü otomatik hariç tut
raw_checkpoints = glob.glob(f"{checkpoints_root}/*slam_asr_tr_tsc*/best_checkpoint")
raw_checkpoints += glob.glob(f"{checkpoints_root}/*/*slam_asr_tr_tsc*/best_checkpoint")

all_tsc_checkpoints = []
for p in raw_checkpoints:
    p_norm = p.replace("\\", "/")
    if "/eski/" in p_norm.lower():
        continue
    if p not in all_tsc_checkpoints:
        all_tsc_checkpoints.append(p)

all_tsc_checkpoints = sorted(
    all_tsc_checkpoints,
    key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p)
)

print(f"[*] Google Drive'da aktif {len(all_tsc_checkpoints)} adet TSC kontrol noktası tespit edildi ('Eski' hariç):\n")
print(f"{'No':<3} | {'Model Etiketi':<22} | {'Model Klasörü':<45} | {'CV Durumu':<14} | {'Mevcut WER / CER'}")
print("-" * 115)

checkpoint_status = []
for idx, ckpt in enumerate(all_tsc_checkpoints, 1):
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    model_tag = extract_model_tag(folder_name)
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    status = "EKSİK (Test Edilecek)"
    metrics = "-"
    if os.path.exists(eval_file):
        try:
            with open(eval_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        res = json.loads(line)
                        wer = res.get("metrics", {}).get("wer") or res.get("wer")
                        cer = res.get("metrics", {}).get("cer") or res.get("cer")
                        if wer is not None:
                            status = "TAMAMLANDI"
                            w_pct = wer * 100 if wer <= 1.0 else wer
                            c_pct = cer * 100 if cer <= 1.0 else cer
                            metrics = f"WER: {w_pct:.2f}%, CER: {c_pct:.2f}%"
        except Exception:
            status = "MEVCUT (Okunamadı)"
    
    checkpoint_status.append({
        "index": idx,
        "model_tag": model_tag,
        "folder_name": folder_name,
        "folder_dir": folder_dir,
        "best_checkpoint": ckpt,
        "status": status,
        "metrics": metrics
    })
    print(f"{idx:<3} | {model_tag:<22} | {folder_name[:45]:<45} | {status:<14} | {metrics}")


In [ ]:
# 5. Eksik Olan Modelleri Sırayla Common Voice Üzerinde Değerlendirme
import subprocess

# Tüm eksikleri otomatik test etmek için True bırakın.
# (Belirli modelleri seçmek isterseniz: target_indices = [3, 5] gibi liste verebilirsiniz)
run_all_missing = True
#target_indices = None
# Sadece eksik ve en kritik 5 modeli seçmek için:
target_indices = [7, 12, 13, 14, 15] 
# 7:  t3-ng-ta-gormez
# 12: t1-tail-aug
# 13: t1-lora
# 14: t1-mlp-lora
# 15: t1-tail-aug-lora

models_to_evaluate = []
for item in checkpoint_status:
    if target_indices is not None:
        if item["index"] in target_indices:
            models_to_evaluate.append(item)
    elif run_all_missing and "EKSİK" in item["status"]:
        models_to_evaluate.append(item)

print(f"[*] Toplam {len(models_to_evaluate)} model değerlendirilecek:\n")
for m in models_to_evaluate:
    print(f"  • [{m['index']}] {m['model_tag']:<20} ({m['folder_name']})")

print("\n" + "=" * 75)

for m in models_to_evaluate:
    folder_dir = m["folder_dir"]
    ckpt_path = m["best_checkpoint"]
    config_path = os.path.join(folder_dir, "hardware_adaptive_turkish.yaml")
    if not os.path.exists(config_path):
        config_path = "config/hardware_adaptive_turkish.yaml"
    
    output_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    print(f"\n[*] Değerlendirme Başlatılıyor: [{m['model_tag']}] {m['folder_name']}")
    print(f"    Checkpoint:  {ckpt_path}")
    print(f"    Config:      {config_path}")
    print(f"    Hedef Çıktı: {output_file}")
    
    cmd = [
        "python", "evaluate.py",
        "--config", config_path,
        "--checkpoint", ckpt_path,
        "--manifest", "data/cv/test.jsonl",
        "--output_file", "temp_eval_cv.jsonl",
        "--language", "tr",
        "--beam_size", "4"
    ]
    
    ret = subprocess.run(cmd)
    if ret.returncode == 0 and os.path.exists("temp_eval_cv.jsonl"):
        !cp temp_eval_cv.jsonl "{output_file}"
        print(f"[✓] Başarılı! Sonuç Drive'a kaydedildi: {output_file}")
    else:
        print(f"[!] {m['folder_name']} değerlendirmesi sırasında bir sorun oluştu.")

print("\n" + "=" * 75)
print("[*] Tüm seçilen modellerin değerlendirmesi tamamlandı!")


In [ ]:
# 6. Nihai Değerlendirme Tablosunu Yazdırma (Master Results Summary)
import json
import os

print("\n" + "=" * 80)
print("        TÜM MODELLERİN MOZILLA COMMON VOICE ÇAPRAZ TEST SONUÇLARI")
print("=" * 80)
print(f"{'Model Etiketi':<22} | {'Klasör Adı':<40} | {'WER (%)':<10} | {'CER (%)':<10}")
print("-" * 80)

for ckpt in all_tsc_checkpoints:
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    model_tag = extract_model_tag(folder_name)
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    wer_str = "-"
    cer_str = "-"
    if os.path.exists(eval_file):
        try:
            with open(eval_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        res = json.loads(line)
                        w = res.get("metrics", {}).get("wer") or res.get("wer")
                        c = res.get("metrics", {}).get("cer") or res.get("cer")
                        if w is not None:
                            w_pct = w * 100 if w <= 1.0 else w
                            wer_str = f"{w_pct:.2f}%"
                        if c is not None:
                            c_pct = c * 100 if c <= 1.0 else c
                            cer_str = f"{c_pct:.2f}%"
        except Exception:
            wer_str = "Hata"
    
    print(f"{model_tag:<22} | {folder_name[:40]:<40} | {wer_str:<10} | {cer_str:<10}")

print("=" * 80)
